# Logistic regression and support vector machines

Logistic regression and support vector machines (SVMs) both answer **yes or no** questions. Both of them draw a **line** between two groups. The difference is in *how* they choose that line, and in *what they tell you* about a new point.

In this notebook you will train both models on the same data, look inside them, work out a prediction by hand, and see where each one does well.

**How to use this notebook**

- Run each cell in order with **Shift + Enter**.
- Cells marked **Exercise** need you to do something. Most just need you to replace `___` with the right value, or answer a question in a text cell.
- Some cells check your answer for you and print *Correct* or *Not quite*.
- If something breaks, use **Kernel > Restart & Run All** and work down from the top again.

## Key words

| Word | Meaning |
|---|---|
| **Feature** | A piece of information we know about each example, such as hours of revision |
| **Label** | The answer we want to predict: here 1 = passed, 0 = failed |
| **Decision boundary** | The line that splits the two groups |
| **Probability** | A chance between 0 and 1. Logistic regression gives you one of these |
| **Support vectors** | The few points closest to the line. An SVM's line depends only on these |
| **Margin** | The gap, or "street", between the line and the closest points on each side |
| **Hyperparameter** | A setting **we** choose before training, rather than something the model learns |

## Part 1: Setting up and the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")   # hide technical warnings so the output is easier to read

We will use **simulated data**: 80 made-up students. For each one we know two features:

- **hours**: how many hours they revised (0 to 10)
- **lectures**: how many of the 12 lectures they attended

and one label: whether they **passed** (1) or **failed** (0). More revision and more lectures make passing more likely, but nothing is guaranteed, just like real life.

In [ ]:
rng = np.random.default_rng(2)      # fixed seed, so everyone gets the same students

hours = np.round(rng.uniform(0, 10, 80), 1)
lectures = rng.integers(0, 13, 80)
chance = 1 / (1 + np.exp(-(1.0 * hours + 0.5 * lectures - 8)))   # hidden rule used to make the data
passed = (rng.random(80) < chance).astype(int)

students = pd.DataFrame({"hours": hours, "lectures": lectures, "passed": passed})
students.head(8)

Before training anything, we put a quarter of the students to one side as a **test set**. The models never see these students while they learn, so we can use them at the end for a fair test.

In [ ]:
X = students[["hours", "lectures"]].to_numpy(dtype=float)
y = students["passed"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print("Training students:", len(X_train))
print("Test students:    ", len(X_test))

In [ ]:
def plot_students(X, y):
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color="tab:blue", s=50, label="Passed")
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color="tab:orange", marker="s", s=50, label="Failed")
    plt.xlabel("Hours of revision")
    plt.ylabel("Lectures attended")

plt.figure(figsize=(7, 5))
plot_students(X_train, y_train)
plt.legend()
plt.title("Training students")
plt.show()

### Exercise 1: think first

Look at the chart. Without writing any code:

a) Would you expect a student who revised for **9 hours** and attended **11 lectures** to pass or fail?

b) What about a student who revised for **1 hour** and attended **2 lectures**?

c) Roughly where would you draw a straight line to split the passes from the fails?

*Write your answer here.*

## Part 2: Logistic regression

Logistic regression works out a **score** for each student:

> z = w1 × hours + w2 × lectures + b

Then it squashes that score into a **probability** between 0 and 1 using the **sigmoid** function:

> probability = 1 / (1 + e^(-z))

If the probability is above 0.5 it predicts **pass**, otherwise **fail**. The numbers w1, w2 and b are learned from the training data.

### Exercise 2: train the model

Fill in the blanks so the model learns from the **training** features and the **training** labels.

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(___, ___)

print("w1 (hours):   ", round(log_reg.coef_[0][0], 2))
print("w2 (lectures):", round(log_reg.coef_[0][1], 2))
print("b:            ", round(log_reg.intercept_[0], 2))

Both weights are positive. What does that tell you about the effect of revising more, or going to more lectures?

*Write your answer here.*

### Exercise 3: ask the model about a new student

A new student revised for **6 hours** and attended **8 lectures**. Fill in the blanks to get the model's probability that they pass.

`predict_proba` gives two numbers: the chance of **fail** and the chance of **pass**. We take the second one with `[0][1]`.

In [ ]:
new_student = [[___, ___]]

chance_of_pass = log_reg.predict_proba(new_student)[0][1]
print("Chance of passing:", round(chance_of_pass, 3))
print("Prediction:", log_reg.predict(new_student)[0])

### Exercise 4: work it out by hand

Now do the same calculation yourself, using the weights the model learned. Fill in the two blanks:

- the **score** z for 6 hours and 8 lectures
- the **sigmoid** that turns z into a probability

In [ ]:
w1 = log_reg.coef_[0][0]
w2 = log_reg.coef_[0][1]
b = log_reg.intercept_[0]

z = ___
probability = ___

print("Score z:    ", round(z, 3))
print("Probability:", round(probability, 3))

In [ ]:
# Check cell: just run this
if abs(probability - log_reg.predict_proba([[6.0, 8.0]])[0][1]) < 0.001:
    print("Correct! Your answer matches the model.")
else:
    print("Not quite. Check your score and your sigmoid.")

## Part 3: Support vector machine

An SVM also draws a straight line, but it chooses the line differently. It looks for the line with the **widest possible street** between the two groups. The points on the edge of the street are called **support vectors**, and they are the only points that decide where the line goes.

### Exercise 5: train the SVM

`SVC` is scikit-learn's SVM. Fill in the blank to ask for a **straight line**. The name of this kind of kernel is `"linear"`.

In [ ]:
svm = SVC(kernel=___)
svm.fit(X_train, y_train)

print("w1 (hours):   ", round(svm.coef_[0][0], 2))
print("w2 (lectures):", round(svm.coef_[0][1], 2))
print("b:            ", round(svm.intercept_[0], 2))

The SVM's numbers are different from logistic regression's, but notice the pattern is similar: hours matter more than lectures, and both are positive.

Here is a function that draws a model's line on the chart. For an SVM it also draws the edges of the street (dashed) and circles the support vectors.

In [ ]:
def draw_model(model, X, y, title):
    xx, yy = np.meshgrid(np.linspace(0, 10, 200), np.linspace(-0.5, 12.5, 200))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    scores = model.decision_function(grid).reshape(xx.shape)

    plt.contourf(xx, yy, scores > 0, alpha=0.12, cmap="coolwarm_r")
    plt.contour(xx, yy, scores, levels=[0], colors="black", linewidths=2)
    if isinstance(model, SVC):
        plt.contour(xx, yy, scores, levels=[-1, 1], colors="grey", linestyles="--")
        sv = model.support_vectors_
        plt.scatter(sv[:, 0], sv[:, 1], s=180, facecolors="none", edgecolors="black", label="Support vectors")
    plot_students(X, y)
    plt.title(title)

plt.figure(figsize=(7, 5))
draw_model(svm, X_train, y_train, "SVM: line, street and support vectors")
plt.legend(loc="upper right", fontsize=8)
plt.show()

### Exercise 6: count the support vectors

The support vectors are stored in `svm.support_vectors_`. Fill in the blank to count them.

In [ ]:
n_support = len(___)
print("Support vectors:", n_support, "out of", len(X_train), "training students")

If you deleted one of the students that is **not** a support vector and trained again, would the line move? Why?

*Write your answer here.*

### Exercise 7: the SVM's score

An SVM does **not** give a probability. Instead it gives a **score** using the same kind of sum:

> score = w1 × hours + w2 × lectures + b

- A score **above 0** means pass, **below 0** means fail.
- A score between **-1 and +1** means the student is **inside the street**, close to the line. These are the less certain predictions.

Fill in the blanks to score our student with 6 hours and 8 lectures, and a second student with **4 hours and 6 lectures**.

In [ ]:
w1 = svm.coef_[0][0]
w2 = svm.coef_[0][1]
b = svm.intercept_[0]

score_a = w1 * 6 + w2 * 8 + b
score_b = ___

print("Student A (6 hours, 8 lectures):", round(score_a, 2))
print("Student B (4 hours, 6 lectures):", round(score_b, 2))

# scikit-learn can do this for us too
print(svm.decision_function([[6.0, 8.0], [4.0, 6.0]]).round(2))

What does the SVM predict for each student? Which one is the SVM less sure about, and how can you tell?

*Write your answer here.*

## Part 4: Side by side

Here are both lines on the same training data.

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
draw_model(log_reg, X_train, y_train, "Logistic regression")

plt.subplot(1, 2, 2)
draw_model(svm, X_train, y_train, "SVM (linear)")

plt.show()

The two lines are very close. That is normal: on simple data like this, both models usually find a similar boundary.

### Exercise 8: the fair test

Now use the **test** students, who neither model has seen. `.score()` gives the accuracy: the fraction of students predicted correctly. Fill in the blanks.

In [ ]:
print("Logistic regression test accuracy:", log_reg.score(___, ___))
print("SVM test accuracy:                ", svm.score(___, ___))

Which model did better on the test students? Does that mean it will always be the better model?

*Write your answer here.*

### Exercise 9: what does each model tell you?

Both models predicted that student A (6 hours, 8 lectures) will pass. Imagine you are a personal tutor deciding which students to contact.

a) What extra information does **logistic regression** give you about student A?

b) What does the **SVM** give you instead?

c) Which would be more useful for a message like *"you have about a 30% chance of passing"*?

*Write your answer here.*

## Part 5: The hyperparameter C

Real data overlaps, so the SVM is allowed to break a few rules: some points may sit inside the street or even on the wrong side. **C** sets how strict it is about this.

- **Small C**: relaxed. A wide street, and many points are allowed inside it.
- **Large C**: strict. A narrow street that tries hard to avoid any mistakes on the training data.

C is a **hyperparameter**. The model does not learn it. **We** choose it before training, and we judge the choice using data the model has not seen. (The weights w1, w2 and b are the opposite: the model learns them.) scikit-learn uses C = 1 if you do not say otherwise.

### Exercise 10: try three values of C

Fill in the blank so the SVM uses the value of C from the loop.

In [ ]:
for c in [0.01, 1, 100]:
    model = SVC(kernel="linear", C=___)
    model.fit(X_train, y_train)
    print("C =", c,
          "| support vectors:", len(model.support_vectors_),
          "| training accuracy:", round(model.score(X_train, y_train), 3),
          "| test accuracy:", round(model.score(X_test, y_test), 3))

In [ ]:
plt.figure(figsize=(12, 4.5))
for i, c in enumerate([0.01, 100]):
    plt.subplot(1, 2, i + 1)
    draw_model(SVC(kernel="linear", C=c).fit(X_train, y_train), X_train, y_train, f"C = {c}")
plt.show()

a) What happens to the number of support vectors as C gets bigger? Use the charts to explain why.

b) Why should we pick C using the **test** accuracy (or other unseen data) and not the training accuracy?

*Write your answer here.*

## Part 6: When a straight line is not enough

So far a straight line has worked well. Here is some data where it cannot: one group sits in a ring around the other. (This is also simulated data.)

In [ ]:
from sklearn.datasets import make_circles

X_c, y_c = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=0)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_c, y_c, test_size=0.25, random_state=0)

plt.figure(figsize=(5, 5))
plt.scatter(X_c[y_c == 1, 0], X_c[y_c == 1, 1], color="tab:blue", s=30, label="Inner group")
plt.scatter(X_c[y_c == 0, 0], X_c[y_c == 0, 1], color="tab:orange", marker="s", s=30, label="Outer ring")
plt.legend()
plt.title("Circles data")
plt.show()

In [ ]:
circle_log_reg = LogisticRegression().fit(Xc_train, yc_train)
circle_svm_line = SVC(kernel="linear").fit(Xc_train, yc_train)

print("Logistic regression test accuracy:", circle_log_reg.score(Xc_test, yc_test))
print("Linear SVM test accuracy:         ", circle_svm_line.score(Xc_test, yc_test))

Both are close to 50%, which is no better than tossing a coin. No straight line can separate a ring from the middle.

This is where SVMs have a trick that plain logistic regression does not: the **kernel trick**. With the `"rbf"` kernel, the SVM can draw a **curved** boundary.

### Exercise 11: switch the kernel

Fill in the blank to use the `"rbf"` kernel.

In [ ]:
circle_svm_curve = SVC(kernel=___).fit(Xc_train, yc_train)

print("RBF SVM test accuracy:", circle_svm_curve.score(Xc_test, yc_test))

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
grid = np.column_stack([xx.ravel(), yy.ravel()])

plt.figure(figsize=(10, 4.5))
for i, (model, name) in enumerate([(circle_log_reg, "Logistic regression (straight line)"), (circle_svm_curve, "RBF SVM (curved line)")]):
    plt.subplot(1, 2, i + 1)
    scores = model.decision_function(grid).reshape(xx.shape)
    plt.contourf(xx, yy, scores > 0, alpha=0.12, cmap="coolwarm_r")
    plt.contour(xx, yy, scores, levels=[0], colors="black", linewidths=2)
    plt.scatter(X_c[y_c == 1, 0], X_c[y_c == 1, 1], color="tab:blue", s=20)
    plt.scatter(X_c[y_c == 0, 0], X_c[y_c == 0, 1], color="tab:orange", marker="s", s=20)
    plt.title(name)
plt.show()

In your own words, why does the RBF SVM do so much better on this data?

*Write your answer here.*

## Part 7 (bonus): real medical data

The breast cancer dataset that comes with scikit-learn has **569 real tumour samples**, each with **30 measurements**. The label says whether the tumour was benign or malignant.

The measurements are on very different scales: some are tiny decimals and some are in the hundreds or thousands. SVMs measure distances, so a feature with big numbers can drown out the others. **Scaling** puts every feature on a similar scale first.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cancer = load_breast_cancer()
Xb_train, Xb_test, yb_train, yb_test = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=0)

print("Largest", cancer.feature_names[9], "value:", cancer.data[:, 9].max().round(3))
print("Largest", cancer.feature_names[23], "value:", cancer.data[:, 23].max().round(0))

### Exercise 12: add scaling

The first model has no scaling. Fill in the blank so the second model puts a `StandardScaler()` in front of the SVM.

In [ ]:
svm_unscaled = SVC()
svm_scaled = make_pipeline(___, SVC())

svm_unscaled.fit(Xb_train, yb_train)
svm_scaled.fit(Xb_train, yb_train)

print("SVM without scaling:", round(svm_unscaled.score(Xb_test, yb_test), 3))
print("SVM with scaling:   ", round(svm_scaled.score(Xb_test, yb_test), 3))

Now try logistic regression with scaling on the same data. Copy the pattern above.

In [ ]:
log_reg_scaled = make_pipeline(StandardScaler(), ___)
log_reg_scaled.fit(Xb_train, yb_train)

print("Logistic regression with scaling:", round(log_reg_scaled.score(Xb_test, yb_test), 3))

*Write your answer here.*

## Summary

| | Logistic regression | SVM |
|---|---|---|
| What it draws | A straight line | The line with the widest street (or a curve with a kernel) |
| Which points matter | Every training point | Only the support vectors |
| What it tells you | A **probability** of each class | A **score**: which side, and how far from the line |
| Curved boundaries? | Not on its own | Yes, with the `"rbf"` kernel |
| Main hyperparameter | C (strength of regularisation) | C (how strict the street is), plus gamma for RBF |
| Needs scaling? | Helps | Very important |

### Final question

A hospital wants a model that tells doctors **how likely** a patient is to have a disease, so they can explain the risk to the patient. Another team has data where the two groups form a curved, swirly shape. Which model would you suggest to each, and why?

*Write your answer here.*